
# Top Features by Best Models (Cases & Trucks)

This notebook pulls the best test-model per department/target from `models/dept*/`, loads the saved final pickles, and surfaces the top 10 features based on model importances. It uses the same 81-feature split files and the department-specific feature sets.


In [7]:

from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import StackingRegressor

BASE_DIR = Path('.')
SPLITS_DIR = BASE_DIR / 'data' / 'processed' / 'timeseries_splits'
MODELS_DIR = BASE_DIR / 'models'


In [8]:

# Load splits
train = pd.read_csv(SPLITS_DIR / 'train_timeseries.csv', parse_dates=['dt'], low_memory=False)
val = pd.read_csv(SPLITS_DIR / 'val_timeseries.csv', parse_dates=['dt'], low_memory=False)
test = pd.read_csv(SPLITS_DIR / 'test_timeseries.csv', parse_dates=['dt'], low_memory=False)

# Feature columns per target (numeric, excluding the target and its lags when inappropriate)
feature_columns_cases = [c for c in train.select_dtypes(include=['number']).columns if c != 'cases']
feature_columns_trucks = [c for c in train.select_dtypes(include=['number']).columns if c != 'trucks' and not c.startswith('cases')]

feature_map = {'cases': feature_columns_cases, 'trucks': feature_columns_trucks}
print('Case features:', len(feature_columns_cases), '| Truck features:', len(feature_columns_trucks))


Case features: 69 | Truck features: 62


In [9]:

# Helper: best test model per dept/target from metrics files

def load_best_models(models_dir=MODELS_DIR):
    rows = []
    for dept_dir in sorted(models_dir.glob('dept*')):
        dept = int(dept_dir.name.replace('dept', ''))
        for target_short, target_label in [('case', 'cases'), ('truck', 'trucks')]:
            path = dept_dir / f'dept{dept}_{target_short}_model_results.csv'
            if not path.exists():
                continue
            df = pd.read_csv(path)
            test_rows = df[df['model'].str.contains('test', na=False)]
            if test_rows.empty:
                continue
            best = test_rows.sort_values('MAPE').iloc[0]
            rows.append({
                'dept': dept,
                'target': target_label,
                'best_model_label': best['model'],
                'best_model_name': best.get('model_name', ''),
                'mape': best['MAPE'],
                'mae': best['MAE'],
                'rmse': best['RMSE'],
                'dir': dept_dir,
            })
    return pd.DataFrame(rows)

best_models = load_best_models()
best_models


,dept,target,best_model_label,best_model_name,mape,mae,rmse,dir
0,41,cases,stacker_cases_test,NaN,0.068079,3.565738,5.274456,models/dept41
1,41,trucks,stacker_trucks_test,NaN,0.044258,0.113611,0.247084,models/dept41
2,6,cases,cat_test,cat,0.069147,3.827758,5.821016,models/dept6
3,6,trucks,stacker_trucks_test,NaN,0.045357,0.116901,0.248318,models/dept6
4,67,cases,cat_test,cat,0.069187,3.866175,5.350469,models/dept67
5,67,trucks,stacker_trucks_test,NaN,0.045066,0.115153,0.247143,models/dept67
6,9,cases,stacker_cases_test,NaN,0.068418,4.046728,6.600023,models/dept9
7,9,trucks,stacker_trucks_test,NaN,0.045187,0.115890,0.247278,models/dept9
8,90,cases,stacker_cases_test,NaN,0.071001,4.168136,5.747895,models/dept90
9,90,trucks,stacker_trucks_test,NaN,0.045522,0.117410,0.249390,models/dept90


In [10]:

# Helper: load final model pickle for a dept/target (falls back to best label if needed)

def load_final_model(dept_dir: Path, target: str):
    # target is 'cases' or 'trucks'
    fname = 'final_case_model.pkl' if target == 'cases' else 'final_truck_model.pkl'
    p = dept_dir / fname
    if p.exists():
        return joblib.load(p), fname
    return None, None


In [11]:

# Helper: extract feature importances

def _from_attr(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        vals = np.asarray(model.feature_importances_)
    elif hasattr(model, 'coef_'):
        vals = np.asarray(model.coef_).ravel()
    else:
        return None
    if len(vals) != len(feature_names):
        # shape mismatch; bail
        return None
    return pd.DataFrame({'feature': feature_names, 'importance': np.abs(vals)})


def get_importances(model, feature_names):
    # Direct importances
    direct = _from_attr(model, feature_names)
    if direct is not None:
        return direct

    # Stacking: blend base estimator importances if available
    if isinstance(model, StackingRegressor):
        frames = []
        if hasattr(model, 'named_estimators_'):
            estimators_items = list(model.named_estimators_.items())
        elif hasattr(model, 'estimators_'):
            estimators_items = [(f'estimator_{i}', est) for i, est in enumerate(model.estimators_)]
        else:
            estimators_items = []

        for est_name, est in estimators_items:
            imp = _from_attr(est, feature_names)
            if imp is not None:
                imp = imp.rename(columns={'importance': f'{est_name}_imp'})
                frames.append(imp)
        if frames:
            merged = frames[0]
            for f in frames[1:]:
                merged = merged.merge(f, on='feature', how='outer')
            merged = merged.fillna(0)
            merged['importance'] = merged.filter(like='_imp').mean(axis=1)
            return merged[['feature', 'importance']]
    return None


In [12]:

# Build top-10 importance tables
importance_rows = []

for _, row in best_models.iterrows():
    dept = row['dept']
    target = row['target']
    dept_dir = row['dir']
    model, source = load_final_model(dept_dir, target)
    if model is None:
        print(f"Skipping dept {dept} {target}: no final model pickle found")
        continue

    features = feature_map[target]
    imps = get_importances(model, features)
    if imps is None:
        print(f"No importances available for dept {dept} {target} ({row['best_model_label']})")
        continue

    top10 = imps.sort_values('importance', ascending=False).head(10).copy()
    top10['dept'] = dept
    top10['target'] = target
    top10['best_model_label'] = row['best_model_label']
    importance_rows.append(top10)

importance_df = pd.concat(importance_rows, ignore_index=True) if importance_rows else pd.DataFrame()
importance_df


,feature,importance,dept,target,best_model_label
0,cases_lag_7,6.465046,41,cases,stacker_cases_test
1,trucks,5.653271,41,cases,stacker_cases_test
2,cases_lag_14,5.174732,41,cases,stacker_cases_test
3,cases_lag_1,2.315393,41,cases,stacker_cases_test
4,cases_ma_7,2.178870,41,cases,stacker_cases_test
...,...,...,...,...,...
95,precip_in,1116.504857,90,trucks,stacker_trucks_test
96,temp_avg_f,1036.501981,90,trucks,stacker_trucks_test
97,gas_price_usd_per_gallon,961.505048,90,trucks,stacker_trucks_test
98,market_area_nbr,937.004653,90,trucks,stacker_trucks_test


In [14]:

# Pivot-friendly tables: top 10 per dept/target
if not importance_df.empty:
    for dept in sorted(importance_df['dept'].unique()):
        for target in ['cases', 'trucks']:
            subset = importance_df[(importance_df['dept']==dept) & (importance_df['target']==target)]
            if subset.empty:
                continue
            print(f"Dept {dept} | {target} | model: {subset['best_model_label'].iloc[0]}")
            display(subset[['feature','importance']].head(10))


Dept 6 | cases | model: cat_test


,feature,importance
20,trucks,17.431522
21,cases_lag_7,16.618515
22,cases_lag_14,14.477598
23,cases_lag_1,6.434414
24,cases_ma_7,4.487025
25,cases_ma_14,4.385068
26,trucks_ma_7,4.053763
27,Crude_Oil_Price,3.648573
28,Initial_Jobless_Claims,3.542775
29,trucks_lag_7,2.573692


Dept 6 | trucks | model: stacker_trucks_test


,feature,importance
30,Crude_Oil_Price,2498.009719
31,temp_max_f,1523.006315
32,temp_min_f,1429.006386
33,trucks_ma_7,1339.024993
34,store_id,1220.505346
35,precip_in,1081.504778
36,temp_avg_f,1054.002996
37,gas_price_usd_per_gallon,1003.505181
38,market_area_nbr,945.004682
39,Initial_Jobless_Claims,751.507250


Dept 9 | cases | model: stacker_cases_test


...

,feature,importance
50,Crude_Oil_Price,1384.735295
51,trucks_ma_7,742.691538
52,temp_max_f,709.827309
53,temp_min_f,654.473773
54,store_id,581.111271
55,precip_in,506.092204
56,temp_avg_f,480.098053
57,gas_price_usd_per_gallon,466.589217
58,market_area_nbr,457.794546
59,state_unemployment_rate,364.165487


Dept 90 | cases | model: stacker_cases_test


,feature,importance
80,trucks,0.348905
81,cases_lag_7,0.145695
82,cases_ma_14,0.079161
83,cases_lag_14,0.071421
84,trucks_lag_7,0.060029
85,cases_ma_7,0.057285
86,trucks_ma_7,0.047105
87,cases_lag_1,0.026292
88,trucks_lag_1,0.018341
89,cases_std_7,0.009959


Dept 90 | trucks | model: stacker_trucks_test


,feature,importance
90,Crude_Oil_Price,2545.509939
91,temp_max_f,1487.006126
92,trucks_ma_7,1436.024635
93,temp_min_f,1418.005982
94,store_id,1194.505396
95,precip_in,1116.504857
96,temp_avg_f,1036.501981
97,gas_price_usd_per_gallon,961.505048
98,market_area_nbr,937.004653
99,Initial_Jobless_Claims,764.507706


In [15]:

# Optional: save to CSV for decks/reports
if not importance_df.empty:
    out_path = BASE_DIR / 'models' / 'top10_feature_importances.csv'
    importance_df.to_csv(out_path, index=False)
    print('Saved top-10 importances to', out_path)


Saved top-10 importances to models/top10_feature_importances.csv
